In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import matplotlib.pyplot as plt
import os
import random
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score
)
import copy
from torch.optim.lr_scheduler import ReduceLROnPlateau
from scipy.signal import butter, filtfilt

all_file_paths = [f"modified_data_Tip_Position_-2mm/corrected_2000RPMfeedspeed30mmmin_IL_bone{i}.csv" for i in range(1, 269)]

random.seed(42)
random.shuffle(all_file_paths)

total_files = len(all_file_paths)
num_train = int(total_files * 0.6)
num_val = int(total_files * 0.2)
num_test = total_files - num_train - num_val

train_paths = all_file_paths[:num_train]
val_paths   = all_file_paths[num_train : num_train + num_val]
test_paths  = all_file_paths[num_train + num_val :]

def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

class PenetrationDataset(Dataset):
    def __init__(
        self,
        file_paths,
        feature_columns=['Fz', 'Torque'],
        target_column='penetration',
        sampling_rate=100,
        fs=10000,
        cutoff=50
    ):
        self.inputs = []
        self.targets = []
        self.feature_columns = feature_columns
        self.target_column = target_column

        for path in file_paths:
            df = pd.read_csv(path)
            df = df[feature_columns + [target_column]].dropna()

            for col in feature_columns:
                filtered = butter_lowpass_filter(df[col].values, cutoff=cutoff, fs=fs)
                df[col] = (filtered - np.mean(filtered)) / np.std(filtered)

            df_downsampled = df.iloc[::100, :].copy()

            input_tensor = torch.tensor(
                df_downsampled[feature_columns].values,
                dtype=torch.float32
            )

            target_tensor = torch.tensor(
                df_downsampled[target_column].values,
                dtype=torch.float32
            )

            self.inputs.append(input_tensor)
            self.targets.append(target_tensor)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

class LSTMClassifier(nn.Module):
    def __init__(self, input_size=2, hidden_size=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        logits = self.output_layer(lstm_out).squeeze(-1)
        return logits

def train_lstm_classifier(model, train_dataset, val_dataset, epochs=300, lr=0.01):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, verbose=True
    )

    best_val_loss = float('inf')
    best_model_state = None

    train_loss_history = []
    val_loss_history = []

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0

        for x_train, y_train in train_dataset:
            x_train = x_train.unsqueeze(0).to(device) 
            y_train = y_train.unsqueeze(0).to(device) 

            optimizer.zero_grad()
            logits = model(x_train) 
            loss = criterion(logits, y_train)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        train_loss_history.append(total_train_loss)

        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for x_val, y_val in val_dataset:
                x_val = x_val.unsqueeze(0).to(device)
                y_val = y_val.unsqueeze(0).to(device)
                logits = model(x_val)
                loss = criterion(logits, y_val)
                total_val_loss += loss.item()

        val_loss_history.append(total_val_loss)
        scheduler.step(total_val_loss)

        if total_val_loss < best_val_loss:
            best_val_loss = total_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            print(
                f"Epoch {epoch+1}: "
                f"Train Loss = {total_train_loss:.4f}, "
                f"Validation Loss = {total_val_loss:.4f} (Best model saved!)"
            )
        else:
            print(
                f"Epoch {epoch+1}: "
                f"Train Loss = {total_train_loss:.4f}, "
                f"Validation Loss = {total_val_loss:.4f}"
            )

    return model, best_model_state, train_loss_history, val_loss_history

def plot_learning_curve(train_loss, val_loss):
    plt.figure(figsize=(10, 5))
    plt.plot(train_loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.xlabel("Epoch")
    plt.ylabel("Loss (BCEWithLogits)")
    plt.title("Learning Curve")
    plt.legend()
    plt.grid(False)
    plt.tight_layout()
    # plt.show()
    plt.savefig("Learning Curve", dpi=330)

def evaluate_critical_region_cls(
    model,
    test_dataset,
    feature_names=['Fz', 'Torque'],
    feature_list=None,
    best_model_state=None,
    device=None,
    n_plot=5,
    window=400
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    model.to(device)
    model.eval()

    acc_total = 0.0
    f1_total = 0.0
    precision_total = 0.0
    recall_total = 0.0
    auc_list = []

    count = 0
    per_seq_f1 = []

    for i, (x, y_true) in enumerate(test_dataset):
        x_input = x.unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x_input)  # (1, seq_len)
            probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()  # (seq_len,)

        x_np = x.numpy()
        y_true_np = y_true.numpy()  # 0/1

        pos_indices = np.where(y_true_np >= 0.5)[0]
        if len(pos_indices) == 0:
            continue

        idx_center = pos_indices[0]
        idx_start = max(0, idx_center - window)
        idx_end = min(len(y_true_np), idx_center + window)

        y_true_slice = y_true_np[idx_start:idx_end]
        y_prob_slice = probs[idx_start:idx_end]
        y_pred_slice = (y_prob_slice >= 0.5).astype(int)
        x_slice = x_np[idx_start:idx_end, :]

        acc = accuracy_score(y_true_slice, y_pred_slice)
        precision = precision_score(y_true_slice, y_pred_slice, zero_division=0)
        recall = recall_score(y_true_slice, y_pred_slice, zero_division=0)
        f1 = f1_score(y_true_slice, y_pred_slice, zero_division=0)

        acc_total += acc
        precision_total += precision
        recall_total += recall
        f1_total += f1
        per_seq_f1.append(f1)

        if len(np.unique(y_true_slice)) == 2:
            auc_list.append(roc_auc_score(y_true_slice, y_prob_slice))

        count += 1

        if count <= n_plot:
            plt.figure(figsize=(14, 6))

            plt.subplot(2, 1, 1)
            plt.plot(y_true_slice, label='True penetration (0/1)', drawstyle='steps-post')
            plt.plot(y_prob_slice, label='Predicted prob (penetration=1)', linestyle='--')
            plt.title(
                f"Sample {i+1} Focused - "
                f"Acc: {acc:.3f}, F1: {f1:.3f}, "
                f"Prec: {precision:.3f}, Rec: {recall:.3f}"
            )
            plt.ylabel("Penetration / Prob")
            plt.ylim(-0.1, 1.1)
            plt.legend()
            plt.grid(True)

            ax1 = plt.subplot(2, 1, 2)
            if feature_list is None:
                feature_list = feature_names
            for name in feature_names:
                idx = feature_list.index(name)
                ax1.plot(x_slice[:, idx], label=name, alpha=0.4)

            ax1.set_ylabel("Input Features")
            plt.tight_layout()
            plt.show()

    avg_acc = acc_total / max(count, 1)
    avg_precision = precision_total / max(count, 1)
    avg_recall = recall_total / max(count, 1)
    avg_f1 = f1_total / max(count, 1)
    avg_auc = np.mean(auc_list) if len(auc_list) > 0 else float('nan')

    print("\n=== Test Summary (Classification in critical region) ===")
    print(f"Average Accuracy : {avg_acc:.4f}")
    print(f"Average Precision: {avg_precision:.4f}")
    print(f"Average Recall   : {avg_recall:.4f}")
    print(f"Average F1-score : {avg_f1:.4f}")
    print(f"Average AUC      : {avg_auc:.4f}")

    return per_seq_f1, avg_acc, avg_f1, avg_auc

from sklearn.metrics import roc_curve, auc

def collect_labels_and_probs(
    model,
    dataset,
    best_model_state=None,
    device=None,
    window=None  # 例: 400 を入れると penetration 直前後 ±400 サンプルに限定
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    model.to(device)
    model.eval()

    all_true = []
    all_prob = []

    with torch.no_grad():
        for x, y_true in dataset:
            x_input = x.unsqueeze(0).to(device)  # (1, seq_len, input_size)
            logits = model(x_input)              # (1, seq_len)
            probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()  # (seq_len,)

            y_true_np = y_true.numpy()  # (seq_len,)

            if window is None:
                y_true_slice = y_true_np
                y_prob_slice = probs
            else:
                pos_indices = np.where(y_true_np >= 0.5)[0]
                if len(pos_indices) == 0:
                    continue  
                idx_center = pos_indices[0]
                idx_start = max(0, idx_center - window)
                idx_end   = min(len(y_true_np), idx_center + window)

                y_true_slice = y_true_np[idx_start:idx_end]
                y_prob_slice = probs[idx_start:idx_end]

            all_true.extend(y_true_slice.tolist())
            all_prob.extend(y_prob_slice.tolist())

    all_true = np.array(all_true)
    all_prob = np.array(all_prob)
    return all_true, all_prob

def plot_roc_curve_cls(
    model,
    dataset,
    best_model_state=None,
    device=None,
    window=None
):
    y_true_all, y_prob_all = collect_labels_and_probs(
        model, dataset,
        best_model_state=best_model_state,
        device=device,
        window=window
    )
    if len(np.unique(y_true_all)) < 2:
        print("ROC を計算するには 0 と 1 の両方が必要ですが、どちらか一方しか含まれていません。")
        return

    fpr, tpr, thresholds = roc_curve(y_true_all, y_prob_all)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
    # plt.xlabel("False Positive Rate")
    # plt.ylabel("True Positive Rate")
    # plt.title("ROC Curve")
    # plt.legend(loc="lower right")
    plt.grid(False)
    plt.tight_layout()
    #plt.show()
    plt.savefig("ROC", dpi=330)

    print(f"AUC: {roc_auc:.4f}")

In [ ]:
feature_columns_zt = ['Fz', 'Torque']

train_dataset_zt = PenetrationDataset(
    train_paths,
    feature_columns=feature_columns_zt,
    target_column='penetration'
)
val_dataset_zt = PenetrationDataset(
    val_paths,
    feature_columns=feature_columns_zt,
    target_column='penetration'
)
test_dataset_zt = PenetrationDataset(
    test_paths,
    feature_columns=feature_columns_zt,
    target_column='penetration'
)

model_zt = LSTMClassifier(input_size=len(feature_columns_zt))

model_zt, best_model_zt_state, train_loss_history_zt, val_loss_history_zt = \
    train_lstm_classifier(model_zt, train_dataset_zt, val_dataset_zt)

plot_learning_curve(train_loss_history_zt, val_loss_history_zt)

seq_f1s, avg_acc, avg_f1, avg_auc = evaluate_critical_region_cls(
    model=model_zt,
    test_dataset=test_dataset_zt,
    feature_names=feature_columns_zt,
    feature_list=feature_columns_zt,
    best_model_state=best_model_zt_state,
    n_plot=5
)

In [1]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plot_roc_curve_cls(
    model_zt,
    test_dataset_zt,
    best_model_state=best_model_zt_state,
    device=device,
    window=None 
)

plot_roc_curve_cls(
    model_zt,
    test_dataset_zt,
    best_model_state=best_model_zt_state,
    device=device,
    window=400 
)

In [3]:
feature_columns_z = ['Fz']

train_dataset_z = PenetrationDataset(
    train_paths,
    feature_columns=feature_columns_z,
    target_column='penetration'
)
val_dataset_z = PenetrationDataset(
    val_paths,
    feature_columns=feature_columns_z,
    target_column='penetration'
)
test_dataset_z = PenetrationDataset(
    test_paths,
    feature_columns=feature_columns_z,
    target_column='penetration'
)

model_z = LSTMClassifier(input_size=len(feature_columns_z))

model_z, best_model_z_state, train_loss_history_z, val_loss_history_z = \
    train_lstm_classifier(model_z, train_dataset_z, val_dataset_z)

plot_learning_curve(train_loss_history_z, val_loss_history_z)

seq_f1s, avg_acc, avg_f1, avg_auc = evaluate_critical_region_cls(
    model=model_z,
    test_dataset=test_dataset_z,
    feature_names=feature_columns_z,
    feature_list=feature_columns_z,
    best_model_state=best_model_z_state,
    n_plot=5
)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plot_roc_curve_cls(
    model_z,
    test_dataset_z,
    best_model_state=best_model_z_state,
    device=device,
    window=None
)

plot_roc_curve_cls(
    model_z,
    test_dataset_z,
    best_model_state=best_model_z_state,
    device=device,
    window=400 
)

In [7]:
feature_columns_t = ['Torque']

train_dataset_t = PenetrationDataset(
    train_paths,
    feature_columns=feature_columns_t,
    target_column='penetration'
)
val_dataset_t = PenetrationDataset(
    val_paths,
    feature_columns=feature_columns_t,
    target_column='penetration'
)
test_dataset_t = PenetrationDataset(
    test_paths,
    feature_columns=feature_columns_t,
    target_column='penetration'
)

model_t = LSTMClassifier(input_size=len(feature_columns_t))  # =1

model_t, best_model_t_state, train_loss_history_t, val_loss_history_t = \
    train_lstm_classifier(model_t, train_dataset_t, val_dataset_t)

plot_learning_curve(train_loss_history_t, val_loss_history_t)

seq_f1s, avg_acc, avg_f1, avg_auc = evaluate_critical_region_cls(
    model=model_t,
    test_dataset=test_dataset_t,
    feature_names=feature_columns_t,
    feature_list=feature_columns_t,
    best_model_state=best_model_t_state,
    n_plot=5
)

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plot_roc_curve_cls(
    model_t,
    test_dataset_t,
    best_model_state=best_model_t_state,
    device=device,
    window=None
)

plot_roc_curve_cls(
    model_t,
    test_dataset_t,
    best_model_state=best_model_t_state,
    device=device,
    window=400 
)